In [1]:
!pip install mne scikit-learn joblib scipy

In [2]:
import os
import zipfile
import joblib
import numpy as np
import mne

from scipy.stats import skew, kurtosis
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix
from google.colab import files

In [3]:
# Upload your zip file: Madhav_Reddy_eeg_data.zip

uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
extract_path = "/content/eeg_data"

with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully.")

Saving Madhav_Reddy_eeg_data.zip to Madhav_Reddy_eeg_data.zip
Dataset extracted successfully.


In [4]:
CHANNELS_19 = [
    "FP1", "FP2", "F7", "F3", "FZ", "F4", "F8",
    "T3", "C3", "CZ", "C4", "T4",
    "T5", "P3", "PZ", "P4", "T6", "O1", "O2"
]

REFERENCE_CHANNELS = [
    "A1", "A2", "A2A1", "A1A2",
    "M1", "M2", "REF", "LE", "AVG",
    "ECG", "EKG", "EOG", "HEOG", "VEOG", "EMG"
]

def clean_channel_name(ch):
    ch = ch.upper()
    ch = ch.replace("EEG", "")
    ch = ch.replace("-LE", "")
    ch = ch.replace("-REF", "")
    ch = ch.replace("-AVG", "")
    ch = ch.replace(".", "")
    ch = ch.replace(" ", "")
    ch = ch.strip()

    mapping = {
        "FP1": "FP1",
        "FP2": "FP2",
        "FZ": "FZ",
        "CZ": "CZ",
        "PZ": "PZ",
        "T7": "T3",
        "T8": "T4",
        "P7": "T5",
        "P8": "T6",
        "A2-A1": "A2A1",
        "A1-A2": "A1A2"
    }

    return mapping.get(ch, ch)

In [5]:
def bandpower(epoch, sfreq, low, high):
    freqs = np.fft.rfftfreq(epoch.shape[-1], d=1 / sfreq)
    fft_vals = np.abs(np.fft.rfft(epoch)) ** 2

    idx = np.logical_and(freqs >= low, freqs <= high)

    if np.sum(idx) == 0:
        return np.zeros(epoch.shape[0])

    return np.mean(fft_vals[:, idx], axis=1)


def extract_features(epoch, sfreq=256):
    features = []

    delta = bandpower(epoch, sfreq, 1, 4)
    theta = bandpower(epoch, sfreq, 4, 8)
    alpha = bandpower(epoch, sfreq, 8, 13)
    beta  = bandpower(epoch, sfreq, 13, 30)
    gamma = bandpower(epoch, sfreq, 30, 40)

    for i in range(epoch.shape[0]):
        ch = epoch[i]

        features.extend([
            np.mean(ch),
            np.std(ch),
            np.var(ch),
            np.min(ch),
            np.max(ch),
            np.sqrt(np.mean(ch ** 2)),
            skew(ch),
            kurtosis(ch),
            delta[i],
            theta[i],
            alpha[i],
            beta[i],
            gamma[i],
            beta[i] / (alpha[i] + 1e-8),
            theta[i] / (alpha[i] + 1e-8)
        ])

    return np.array(features)

In [6]:
def process_edf(file_path, label, target_sfreq=256):
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)

    rename_dict = {ch: clean_channel_name(ch) for ch in raw.ch_names}
    raw.rename_channels(rename_dict)

    raw.drop_channels(
        [ch for ch in raw.ch_names if ch in REFERENCE_CHANNELS],
        on_missing="ignore"
    )

    missing_channels = [ch for ch in CHANNELS_19 if ch not in raw.ch_names]

    if missing_channels:
        print("Skipping file:", file_path)
        print("Missing channels:", missing_channels)
        return [], []

    raw.pick_channels(CHANNELS_19, ordered=True)

    raw.resample(target_sfreq, verbose=False)
    raw.filter(1, 40, fir_design="firwin", verbose=False)
    raw.notch_filter(50, verbose=False)

    data = raw.get_data()

    epoch_samples = 2 * target_sfreq

    X_file = []
    y_file = []

    for start in range(0, data.shape[1] - epoch_samples + 1, epoch_samples):
        epoch = data[:, start:start + epoch_samples]

        feature_vector = extract_features(epoch, sfreq=target_sfreq)

        X_file.append(feature_vector)
        y_file.append(label)

    return X_file, y_file

In [7]:
# Update this only if your extracted folder name is different

BASE_PATH = "/content/eeg_data/Madhav_Reddy_eeg_data"

FOLDER_LABELS = {
    "Eyes_Closed": 0,
    "Task": 1
}

X = []
y = []

for folder, label in FOLDER_LABELS.items():
    folder_path = os.path.join(BASE_PATH, folder)

    for file in os.listdir(folder_path):
        if file.lower().endswith(".edf"):
            file_path = os.path.join(folder_path, file)

            print("Processing:", file_path)

            X_file, y_file = process_edf(file_path, label)

            X.extend(X_file)
            y.extend(y_file)

X = np.array(X)
y = np.array(y)

print("Before balancing:")
print("Total epochs:", len(y))
print("Rest epochs:", np.sum(y == 0))
print("Task epochs:", np.sum(y == 1))
print("Feature size:", X.shape[1])

Processing: /content/eeg_data/Madhav_Reddy_eeg_data/Eyes_Closed/Madhav reddy 01.000.02 AGE 21  EC.edf
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Processing: /content/eeg_data/Madhav_Reddy_eeg_data/Eyes_Closed/Madhav reddy 01.000.04 AGE 21  EC.edf
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Processing: /content/eeg_data/Madhav_Reddy_eeg_data/Task/Madhav reddy 01.000.03 AGE 21  TASK_1.edf
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Before balancing:
Total epochs: 313
Rest epochs: 250
Task epochs: 63
Feature size: 285


In [8]:
# Balance Rest and Task epochs equally

rest_idx = np.where(y == 0)[0]
task_idx = np.where(y == 1)[0]

min_count = min(len(rest_idx), len(task_idx))

np.random.seed(42)

rest_idx_balanced = np.random.choice(rest_idx, min_count, replace=False)
task_idx_balanced = np.random.choice(task_idx, min_count, replace=False)

balanced_idx = np.concatenate([rest_idx_balanced, task_idx_balanced])
np.random.shuffle(balanced_idx)

X = X[balanced_idx]
y = y[balanced_idx]

print("After balancing:")
print("Rest epochs:", np.sum(y == 0))
print("Task epochs:", np.sum(y == 1))
print("Final X shape:", X.shape)
print("Final y shape:", y.shape)

After balancing:
Rest epochs: 63
Task epochs: 63
Final X shape: (126, 285)
Final y shape: (126,)


In [9]:
# Train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training samples:", len(y_train))
print("Testing samples:", len(y_test))

Training samples: 94
Testing samples: 32


In [10]:
# Train Random Forest

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [11]:
# Evaluate model

y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Accuracy :", round(accuracy * 100, 2), "%")
print("Precision:", round(precision * 100, 2), "%")
print("Recall   :", round(recall * 100, 2), "%")
print("F1-Score :", round(f1 * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 78.12 %
Precision: 71.43 %
Recall   : 93.75 %
F1-Score : 81.08 %

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.62      0.74        16
           1       0.71      0.94      0.81        16

    accuracy                           0.78        32
   macro avg       0.81      0.78      0.78        32
weighted avg       0.81      0.78      0.78        32


Confusion Matrix:
[[10  6]
 [ 1 15]]


In [13]:
# Save RF model

model_package = {
    "model": rf_model,
    "channels": CHANNELS_19,
    "sfreq": 256,
    "epoch_seconds": 2,
    "feature_count": X.shape[1],
    "features_per_channel": 15,
    "class_names": ["Eyes Closed / No Stress", "Task / Stress"],
    "purpose": "Research, study, and comparison only"
}

joblib.dump(model_package, "random_forest_live_19ch.pkl")

print("Saved model: random_forest_live_19ch.pkl")

Saved model: random_forest_live_19ch.pkl


In [14]:
# Download model

files.download("random_forest_live_19ch.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>